

---

# Surface material detection using Full Scene imagery

In [ ]:
PROMPT = """Act as a civil engineering material analyst specializing in road surfaces. Your task is to analyze the provided Google Street View image and accurately identify the dominant road surface material present
Your analysis must follow these steps to ensure accuracy:
1.  **Identify Road Surface Area**: Clearly delineate the primary road surface in the image, ignoring sidewalks, shoulders, or surrounding terrain.
2.  **Visual Evidence Extraction**: For the identified road surface, describe the visual cues that indicate its material type. Focus specifically on:
    *   **Texture & Micro-structure**: Look for characteristics such as:
        *   **Paved**: Smooth, uniform, often dark or light grey. May show aggregate, cracks, patches, or lane markings. If asphalt, a granular, somewhat coarse texture. If concrete, a finer, often broom-finished texture with expansion joints.
        *   **Gravel**: Loose, irregularly shaped stones of varying sizes with visible gaps and an uneven surface. Often exhibits tire tracks or displacement.
        *   **Mud**: Soft, wet, or dried soil, often with ruts, puddles, or deep impressions from vehicles. Can vary widely in color and consistency.
        *   **Dirt**: Unpaved, dry, compacted soil. Less uniform than paved, but more stable than mud, often exhibiting dust or tire marks.
    *   **Reflectance & Specularity**: Observe how light interacts with the surface. Is it matte (dirt, some mud), somewhat reflective (wet mud, new asphalt), or does it show clear highlights (wet paved roads, standing water)?
    *   **Contextual Cues**: Consider environmental factors such as surrounding vegetation, drainage, and road infrastructure (e.g., presence of road signs, guardrails nearby, but not directly on the road surface).
**Output Format**:
Provide your findings in a structured JSON format:
{
  "road_surface_material": "[Material classification: Paved (Asphalt/Concrete), Gravel, Mud, Dirt, or Other]",
  "confidence_score": "[0-100%]",
  "visual_reasoning": "[1-3 sentences describing specific visual evidence supporting the classification, e.g., 'Surface exhibits uniform dark gray color with visible aggregate and clear lane markings, consistent with asphalt pavement.']",
  "image_url": "{image_url}"
}
**Note:**
**Important Considerations:**
*   Focus exclusively on the material directly comprising the main driving surface.
*   Ignore temporary conditions like standing water or debris unless they are definitive indicators of the underlying road material.
*   If the material is ambiguous or mixed, classify based on the dominant type and note ambiguity in reasoning."""

In [ ]:
import pandas_gbq
import vertexai
from vertexai.preview.generative_models import GenerativeModel, Part
import urllib.parse

# Query the BigQuery table
project_id = "imagery-insights-trial-491916" #@param {type:"string"}
track_id = 't1:K1H4CwEof7OgJXBAkjj0Cg:5001ee' #@param {type:"string"}
dataset_id = 'woolpert_pilot_dcr' #@param {type:"string"}
observations_table_name = 'observations' #@param {type:"string"}
urls_table_name = 'urls' #@param {type:"string"}

sql_query = f"""
SELECT
  t2.signedUrl
FROM
  `{project_id}`.svfs_pilot_dcr.`{observations_table_name}` AS t1
INNER JOIN
  `{project_id}`.`{dataset_id}`.`{urls_table_name}` AS t2
ON
  t1.observation2 = t2.observationId
WHERE
  t1.trackId = '{track_id}'
"""
df = pandas_gbq.read_gbq(sql_query, project_id, dialect="standard")

# Get the GCS URL
if not df.empty:
    gcs_url = df['signedUrl'].iloc[0]
    print(f"GCS URL (from BigQuery): {gcs_url}")

    # Decode the GCS URL fully first
    fully_decoded_gcs_url = urllib.parse.unquote(gcs_url)
    print(f"Fully Decoded GCS URL: {fully_decoded_gcs_url}")

    # Now, re-encode only the path component of the URL for characters like ':'
    parsed_url = urllib.parse.urlparse(fully_decoded_gcs_url)
    # Encode path, keeping '/' safe (not encoded)
    encoded_path = urllib.parse.quote(parsed_url.path, safe='/')
    # Reconstruct the URL with the encoded path
    final_gcs_url_https = urllib.parse.urlunparse(parsed_url._replace(path=encoded_path))
    print(f"Final Encoded GCS URL (HTTPS): {final_gcs_url_https}")

    # Convert to gs:// URI for Vertex AI if applicable
    if final_gcs_url_https.startswith("https://storage.googleapis.com/"):
        gcs_uri_for_vertex_ai = final_gcs_url_https.replace("https://storage.googleapis.com/", "gs://")
        # Need to handle the bucket name being part of the path in the gs:// scheme
        # Example: https://storage.googleapis.com/bucket/path/to/object -> gs://bucket/path/to/object
        # parsed_url.netloc is 'storage.googleapis.com'
        # parsed_url.path starts with '/bucket/path/to/object'
        # We need 'gs://bucket/path/to/object'

        # Extract bucket and path from the original decoded URL to form gs:// URI
        path_parts = parsed_url.path.split('/', 2)
        if len(path_parts) > 2:
            bucket_name = path_parts[1]
            object_path = path_parts[2]
            gcs_uri_for_vertex_ai = f"gs://{bucket_name}/{object_path}"
        else:
            # Fallback if path structure is not as expected, keep https
            gcs_uri_for_vertex_ai = final_gcs_url_https
    else:
        gcs_uri_for_vertex_ai = final_gcs_url_https

    print(f"Final GCS URI for Vertex AI: {gcs_uri_for_vertex_ai}")

    try:
        # Initialize Vertex AI
        vertexai.init(project=project_id, location="us-central1")
        model_name = "gemini-2.5-flash" #@param {type:"string"}
        model = GenerativeModel(model_name)

        # Prepare the prompt and image for the model using Part.from_uri()
        prompt = PROMPT # Use the PROMPT variable from the earlier cell
        image_part = Part.from_uri(uri=gcs_uri_for_vertex_ai, mime_type="image/jpeg")

        # Send to Gemini 3 model
        response = model.generate_content([image_part, prompt])
        print("Gemini  Model Response:")
        print(response.text)

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
else:
    print("No URL found for the given trackId.")

Downloading: 100%|██████████|
GCS URL (from BigQuery): https://storage.googleapis.com/everything_imagery/full_scene_home_depot/o1:wKmVtEQ9LqEA5FGDmM1buA_2:5001ee.jpg
Fully Decoded GCS URL: https://storage.googleapis.com/everything_imagery/full_scene_home_depot/o1:wKmVtEQ9LqEA5FGDmM1buA_2:5001ee.jpg
Final Encoded GCS URL (HTTPS): https://storage.googleapis.com/everything_imagery/full_scene_home_depot/o1%3AwKmVtEQ9LqEA5FGDmM1buA_2%3A5001ee.jpg
Final GCS URI for Vertex AI: gs://everything_imagery/full_scene_home_depot/o1:wKmVtEQ9LqEA5FGDmM1buA_2:5001ee.jpg
Gemini 3 Model Response:
```json
{
  "road_surface_material": "Paved (Concrete)",
  "confidence_score": "95%",
  "visual_reasoning": "The road surface is light grey and exhibits a uniform, rigid appearance with distinct linear patterns or textures running across its width, characteristic of a broom-finished or aged concrete surface. It lacks the dark, granular texture of asphalt or the loose, irregular nature of gravel or dirt.",
  "ima